# WMT14 En→De — PE Comparison (RoPE / AsRope2 / AsRope3)

Encoder-decoder transformer (6 enc + 6 dec, d=384, h=6, ffn=1536, ~50M params).
Pipeline:
1. Download WMT14 → `raw_data/wmt14/`
2. Tokenize with `Helsinki-NLP/opus-mt-en-de` → `processed_data_wmt14/tokenized/`
3. Sanity check — tiny subset, 200 steps, each PE type
4. Full training — 3 PE types on (up to 5M-capped) WMT14 train
5. Eval on newstest2014 — BLEU / chrF / TER / BLEU-by-length

**Metrics tracked:** BLEU, chrF, TER, validation loss, training time, BLEU by sentence length.


In [ ]:
import os, datetime, pathlib
os.chdir('/home/m240479cs/projects/Neur')
PY = '/home/m240479cs/projects/Neur/.venv/bin/python'

TRAIN_TOK = 'processed_data_wmt14/tokenized/train_wmt14_en_de.pt'
VAL_TOK   = 'processed_data_wmt14/tokenized/val_wmt14_en_de.pt'
TEST_TSV  = 'raw_data/wmt14/test.tsv'

PE_TYPES = ['rope', 'asrope2', 'asrope3']

print('cwd:', os.getcwd())
print('session:', datetime.datetime.now())
!{PY} -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


---
## 1. Download WMT14
Fetches train (~4.5M), newstest2013 val, newstest2014 test. Skips if files already exist.

In [ ]:
import datetime; print('START step1:', datetime.datetime.now())
!{PY} -m pipeline.step1_download
print('END step1:', datetime.datetime.now())

for p in ['raw_data/wmt14/train.tsv','raw_data/wmt14/val.tsv','raw_data/wmt14/test.tsv']:
    f = pathlib.Path(p)
    if f.exists():
        n = sum(1 for _ in open(f))
        print(f'  {p}: {n:,} lines ({f.stat().st_size/1e6:.0f} MB)')
    else:
        print(f'  MISSING: {p}')


---
## 2. Tokenize WMT14 (pair format)

Each example stores separate src/tgt int32 buffers (no `<sep>` concatenation).
Collator adds BOS/EOS + pads at batch time.

In [ ]:
# Full tokenization (~30 min). For a quick sanity run, pass --max-train 200000.
import datetime; print('START step2:', datetime.datetime.now())
!{PY} -m pipeline.step2_tokenize --max-seq-len 128
print('END step2:', datetime.datetime.now())

import torch, pathlib
for p in [TRAIN_TOK, VAL_TOK, 'processed_data_wmt14/tokenized/test_wmt14_en_de.pt']:
    f = pathlib.Path(p)
    if f.exists():
        d = torch.load(f, weights_only=False)
        m = d['meta']
        print(f"  {p}: n={m['n_examples']:,}  src_p99={m['src_len_p99']:.0f}  "
              f"tgt_p99={m['tgt_len_p99']:.0f}  size={f.stat().st_size/1e6:.0f}MB")
    else:
        print(f'  MISSING: {p}')


---
## 3. Sanity check (end-to-end)

Runs tiny training + eval for each PE type to verify the stack works before the big training.

- 200 steps
- batch 16
- evaluates on a 200-example subset
- saves checkpoints under `outputs/checkpoints/<pe>_sanity/`


In [ ]:
# Sanity: RoPE — 200 steps
import datetime; print('SANITY rope:', datetime.datetime.now())
!{PY} -m pipeline.step3_train \
    --pe-type rope --run-name rope_sanity \
    --tokenized-train {TRAIN_TOK} --tokenized-val {VAL_TOK} \
    --num-steps 200 --eval-every 100 --val-subset 200 \
    --batch-size 16 --max-seq-len 128 --dataloader-workers 0 \
    --no-use-compile
print('DONE rope sanity:', datetime.datetime.now())


In [ ]:
# Sanity: AsRope2 — 200 steps
import datetime; print('SANITY asrope2:', datetime.datetime.now())
!{PY} -m pipeline.step3_train \
    --pe-type asrope2 --run-name asrope2_sanity \
    --tokenized-train {TRAIN_TOK} --tokenized-val {VAL_TOK} \
    --num-steps 200 --eval-every 100 --val-subset 200 \
    --batch-size 16 --max-seq-len 128 --dataloader-workers 0 \
    --no-use-compile
print('DONE asrope2 sanity:', datetime.datetime.now())


In [ ]:
# Sanity: AsRope3 — 200 steps
import datetime; print('SANITY asrope3:', datetime.datetime.now())
!{PY} -m pipeline.step3_train \
    --pe-type asrope3 --run-name asrope3_sanity \
    --tokenized-train {TRAIN_TOK} --tokenized-val {VAL_TOK} \
    --num-steps 200 --eval-every 100 --val-subset 200 \
    --batch-size 16 --max-seq-len 128 --dataloader-workers 0 \
    --no-use-compile
print('DONE asrope3 sanity:', datetime.datetime.now())


In [ ]:
# Sanity eval: run step4 on each sanity checkpoint against a 200-line slice of newstest2014
# (quick pass — just verifies decode + metrics run end to end).
import datetime, subprocess
!head -n 200 raw_data/wmt14/test.tsv > /tmp/test_slice.tsv
for pe in PE_TYPES:
    print(f'--- sanity eval {pe} ---')
    !{PY} -m pipeline.step4_eval \
        --checkpoint outputs/checkpoints/{pe}_sanity/best.pt \
        --eval-tsv /tmp/test_slice.tsv \
        --run-name {pe}_sanity_eval \
        --batch-size 32 --max-new-tokens 64


---
## 4. Full training

3 PE types × ~5M pair cap (WMT14 has ~4.5M — cap is effectively the full set).
Default: 100K steps, batch 64 (effective 64 — no grad accum), max_seq_len 128, bf16 + torch.compile.

Per run: ~3–4 hrs on A100.

In [ ]:
import datetime; print('START rope_de:', datetime.datetime.now())
!{PY} -m pipeline.step3_train \
    --pe-type rope --run-name rope_de \
    --tokenized-train {TRAIN_TOK} --tokenized-val {VAL_TOK} \
    --num-steps 100000 --eval-every 5000 --val-subset 500 \
    --batch-size 64 --learning-rate 5e-4 --warmup-ratio 0.05 \
    --max-seq-len 128 --dropout 0.1 --seed 42
print('END rope_de:', datetime.datetime.now())


In [ ]:
import datetime; print('START asrope2_de:', datetime.datetime.now())
!{PY} -m pipeline.step3_train \
    --pe-type asrope2 --run-name asrope2_de \
    --tokenized-train {TRAIN_TOK} --tokenized-val {VAL_TOK} \
    --num-steps 100000 --eval-every 5000 --val-subset 500 \
    --batch-size 64 --learning-rate 5e-4 --warmup-ratio 0.05 \
    --max-seq-len 128 --dropout 0.1 --seed 42
print('END asrope2_de:', datetime.datetime.now())


In [ ]:
import datetime; print('START asrope3_de:', datetime.datetime.now())
!{PY} -m pipeline.step3_train \
    --pe-type asrope3 --run-name asrope3_de \
    --tokenized-train {TRAIN_TOK} --tokenized-val {VAL_TOK} \
    --num-steps 100000 --eval-every 5000 --val-subset 500 \
    --batch-size 64 --learning-rate 5e-4 --warmup-ratio 0.05 \
    --max-seq-len 128 --dropout 0.1 --seed 42
print('END asrope3_de:', datetime.datetime.now())


In [ ]:
# Training status — shows val loss, best step, training time per PE
import json, pathlib, datetime

print(f"status @ {datetime.datetime.now()}")
print(f"{'Run':<14} {'Status':<10} {'Best Val':>10} {'Best Step':>10} {'Time (hr)':>10} {'Params':>14}")
print('-' * 72)
for pe in PE_TYPES:
    name = f'{pe}_de'
    s = pathlib.Path('outputs/logs') / name / 'run_summary.json'
    ck = pathlib.Path('outputs/checkpoints') / name / 'best.pt'
    if s.exists():
        d = json.loads(s.read_text())
        print(f"{name:<14} {'DONE':<10} {d['best_val_loss']:>10.4f} {d['best_step']:>10} "
              f"{d['total_time_sec']/3600:>10.2f} {d.get('n_params',0):>14,}")
    elif ck.exists():
        print(f"{name:<14} {'PARTIAL':<10} {'—':>10} {'—':>10} {'—':>10} {'—':>14}")
    else:
        print(f"{name:<14} {'PENDING':<10} {'—':>10} {'—':>10} {'—':>10} {'—':>14}")


---
## 5. Evaluation on newstest2014

Full-benchmark eval for each PE type. Writes BLEU / chrF / TER + BLEU-by-source-length
to `outputs/metrics/<pe>_de_eval/eval_summary.json`.

In [ ]:
import datetime; print('EVAL rope_de:', datetime.datetime.now())
!{PY} -m pipeline.step4_eval \
    --checkpoint outputs/checkpoints/rope_de/best.pt \
    --run-name rope_de_eval \
    --eval-tsv {TEST_TSV} --batch-size 32
print('DONE rope_de eval:', datetime.datetime.now())


In [ ]:
import datetime; print('EVAL asrope2_de:', datetime.datetime.now())
!{PY} -m pipeline.step4_eval \
    --checkpoint outputs/checkpoints/asrope2_de/best.pt \
    --run-name asrope2_de_eval \
    --eval-tsv {TEST_TSV} --batch-size 32
print('DONE asrope2_de eval:', datetime.datetime.now())


In [ ]:
import datetime; print('EVAL asrope3_de:', datetime.datetime.now())
!{PY} -m pipeline.step4_eval \
    --checkpoint outputs/checkpoints/asrope3_de/best.pt \
    --run-name asrope3_de_eval \
    --eval-tsv {TEST_TSV} --batch-size 32
print('DONE asrope3_de eval:', datetime.datetime.now())


---
## 6. Results table

In [ ]:
import json, pathlib, datetime

PE_TYPES = ['rope', 'asrope2', 'asrope3']

print(f'Generated: {datetime.datetime.now()}\n')

# Training summary
print('TRAINING')
print(f"{'PE':<10} {'BestVal':>9} {'Step':>7} {'Time(hr)':>9} {'Tok/s':>8} {'Params':>12}")
print('-' * 60)
for pe in PE_TYPES:
    s = pathlib.Path(f'outputs/logs/{pe}_de/run_summary.json')
    if not s.exists():
        print(f'{pe:<10}  (no training summary)')
        continue
    d = json.loads(s.read_text())
    print(f"{pe:<10} {d['best_val_loss']:>9.4f} {d['best_step']:>7} "
          f"{d['total_time_sec']/3600:>9.2f} {d.get('tokens_per_sec_avg',0):>8.0f} "
          f"{d.get('n_params',0):>12,}")

print('\nEVAL (newstest2014)')
print(f"{'PE':<10} {'BLEU':>7} {'chrF':>7} {'TER':>7} {'N':>6}")
print('-' * 46)
for pe in PE_TYPES:
    s = pathlib.Path(f'outputs/metrics/{pe}_de_eval/eval_summary.json')
    if not s.exists():
        print(f'{pe:<10}  (no eval)')
        continue
    d = json.loads(s.read_text())
    ov = d['overall']
    print(f"{pe:<10} {ov['bleu']:>7.2f} {ov['chrf']:>7.2f} {ov['ter']:>7.2f} {ov['n']:>6}")

print('\nBLEU BY SOURCE LENGTH')
header = ['range'] + PE_TYPES
print(f"{'range':<10} " + ''.join(f"{p:>10}" for p in PE_TYPES))
print('-' * (10 + 10*len(PE_TYPES)))
# assume all have same buckets
ranges = None
per_pe = {}
for pe in PE_TYPES:
    s = pathlib.Path(f'outputs/metrics/{pe}_de_eval/eval_summary.json')
    if not s.exists():
        per_pe[pe] = None
        continue
    d = json.loads(s.read_text())
    per_pe[pe] = d['by_src_length']
    if ranges is None:
        ranges = [b['range'] for b in d['by_src_length']]
if ranges:
    for i, r in enumerate(ranges):
        row = f'{r:<10} '
        for pe in PE_TYPES:
            buck = per_pe[pe]
            if buck is None:
                row += f"{'—':>10}"
            else:
                v = buck[i]['bleu']
                row += f"{('—' if v is None else f'{v:.2f}'):>10}"
        print(row)
